# Case Study 2 — full pipeline (run top to bottom)

This one notebook runs everything on the university server: build the corpus, embed and index, generate, judge, score, and show the four-bucket result.

**Two switches, set in section 2:**
- **Generator** — the free dev model (Gemini) to shake out bugs now, or the frozen `claude-sonnet-4-6` for the real graded run.
- **Judge** — `stub` (offline, instant) for dev, or the frozen 70B open model via vLLM on this GPU for the real run.

Free-model runs are for **debugging the pipeline, not results**. The numbers that go in the manuscript use the frozen generator + the 70B judge.

Run the cells in order. Sections 3 and 4 build the retrieval store (once). Section 6 generates, section 7 judges and scores, section 8 shows the result.

## 0. Environment probe
Tells us what this server can do. Run it first.

In [9]:
import sys, os, subprocess, platform, urllib.request
print("python:", sys.version.split()[0], "|", platform.platform())
print("cwd:", os.getcwd())
if not os.path.exists("src/run_generation.py"):
    print("!! Run this notebook from the repo ROOT (the folder with src/, config/, test_set.jsonl).")

def check_internet(url="https://pypi.org", timeout=5):
    try:
        urllib.request.urlopen(url, timeout=timeout); return True
    except Exception as e:
        print("  internet check failed:", e); return False

HAS_INTERNET = check_internet()
print("internet:", HAS_INTERNET)

HAS_GPU = False
try:
    import torch
    HAS_GPU = torch.cuda.is_available()
    if HAS_GPU:
        p = torch.cuda.get_device_properties(0)
        print("GPU:", torch.cuda.get_device_name(0),
              f"| VRAM {p.total_memory/1e9:.0f} GB | count {torch.cuda.device_count()}")
    else:
        print("GPU: torch present but no CUDA device visible")
except Exception as e:
    print("GPU: torch not importable yet (install deps in section 1) ->", e)
print(f"\nSUMMARY  internet={HAS_INTERNET}  gpu={HAS_GPU}")

python: 3.11.13 | Linux-6.8.0-138-generic-x86_64-with-glibc2.35
cwd: /home/jovyan/case_study2
internet: True
GPU: NVIDIA RTX A6000 | VRAM 51 GB | count 2

SUMMARY  internet=True  gpu=True


## 1. Install dependencies (run once, needs internet)
vLLM for the 70B judge is heavy and installed later, only when you switch the judge on.

In [26]:
if HAS_INTERNET:
    subprocess.run([sys.executable,"-m","pip","install","-q","-r","requirements.txt"], check=False)
    subprocess.run([sys.executable,"-m","pip","install","-q","openai"], check=False)
    print("core deps installed")
else:
    print("No internet here: install where there is internet, or pre-stage wheels.")


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python3 -m pip install --upgrade pip


core deps installed



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python3 -m pip install --upgrade pip


## 2. Config — the only knobs

For the **free dev run** (default): Gemini generator + stub judge. Paste your free Google AI Studio key below.

For the **real graded run**: set `GEN_PROVIDER="anthropic"`, `GEN_MODEL="claude-sonnet-4-6"`, paste `ANTHROPIC_API_KEY`, set `JUDGE="vllm"`, and `RUN_FULL=True`.

In [ ]:
# ---------- GENERATOR ----------
GEN_PROVIDER = "openai_compatible"      # "anthropic" for the frozen graded run
GEN_MODEL    = "gemini-3.6-flash"       # "claude-sonnet-4-6" for the frozen run
GEN_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GEN_KEY_ENV  = "GEMINI_API_KEY"
os.environ["GEMINI_API_KEY"] = os.environ.get("GEMINI_API_KEY", "")   # <-- paste free key
# os.environ["ANTHROPIC_API_KEY"] = ""   # <-- paste for the frozen run

# ---------- JUDGE ----------
JUDGE          = "vllm"                  # "vllm" for the real 70B judge on this GPU
JUDGE_MODEL    = "Qwen/Qwen2.5-72B-Instruct-AWQ"   # or meta-llama/Llama-3.3-70B-Instruct
JUDGE_BASE_URL = "http://localhost:8000/v1"

# ---------- RUN SIZE ----------
RUN_FULL = False    # False = 8-row sanity per config; True = full 215 x 3

def sh(cmd):
    print("$", " ".join(cmd) if isinstance(cmd, list) else cmd)
    r = subprocess.run(cmd, shell=not isinstance(cmd, list), capture_output=True, text=True)
    if r.stdout: print(r.stdout[-6000:])
    if r.returncode != 0 and r.stderr: print("STDERR:\n", r.stderr[-4000:])
    return r.returncode

print("generator:", GEN_PROVIDER, GEN_MODEL, "| judge:", JUDGE, "| full run:", RUN_FULL)

generator: openai_compatible gemini-3.6-flash | judge: vllm | full run: False


## 3. Corpus (400 chunks from the frozen guidelines)
Uses the committed corpus if present; only rebuilds if missing (rebuild needs poppler/pdftotext).

In [14]:
CHUNKS = "results/corpus_chunks.jsonl"
PDF = "data/guidelines/Draft_Guidelines_on_the_classification_of_high_risk_AI_Annex_III.pdf"
if os.path.exists(CHUNKS):
    print("corpus present (committed):", sum(1 for _ in open(CHUNKS)), "chunks — skip rebuild")
else:
    sh([sys.executable, "src/build_corpus.py", PDF, CHUNKS])
    print("chunks:", sum(1 for _ in open(CHUNKS)))

corpus present (committed): 400 chunks — skip rebuild


## 4. Embed + index (downloads bge model, needs internet, builds Chroma)
The vector store is not in git, so build it here once.

In [15]:
sh([sys.executable, "src/embed_and_index.py", "config/pipeline.yaml"])

$ /usr/bin/python3 src/embed_and_index.py config/pipeline.yaml
loaded 400 chunks
embedded -> (400, 768)
persisted 400 vectors -> results/chroma/eu_ai_act_guidelines



0

## 5. Retrieval quality check (optional, no API needed)
Should show Hit@5 around 0.83.

In [16]:
sh([sys.executable, "src/retrieval_eval.py", "config/pipeline.yaml"])

$ /usr/bin/python3 src/retrieval_eval.py config/pipeline.yaml

Retrieval quality over 215 rows
  Hit@5   0.833   (PRIMARY)
  Hit@10  0.898
  Recall@5  0.246   Recall@10 0.374
  MRR     0.633
  wrote results/retrieval_eval.json and results/retrieval_eval.md



0

In [20]:
# patch: ride out the free-tier 5-requests-per-minute limit
import pathlib
p = pathlib.Path("src/run_generation.py")
s = p.read_text()
s = s.replace("def call_generator(client, kind, gen_cfg, system, user, max_retries=4):",
              "def call_generator(client, kind, gen_cfg, system, user, max_retries=8):")
s = s.replace("            time.sleep(2 ** attempt)",
              "            time.sleep(15)")
p.write_text(s)
print("patched: 8 retries, 15s wait on rate-limit")

patched: 8 retries, 15s wait on rate-limit


## 6. Generation
Writes one file per config to results/runs/. Sanity (8 rows) unless RUN_FULL=True.

In [21]:
gen_args = [sys.executable, "src/run_generation.py",
            "--provider", GEN_PROVIDER, "--model", GEN_MODEL,
            "--base-url", GEN_BASE_URL, "--api-key-env", GEN_KEY_ENV]
if not RUN_FULL:
    gen_args += ["--limit", "8"]
sh(gen_args)

import glob, json
for f in sorted(glob.glob("results/runs/*.jsonl")):
    rows = [json.loads(l) for l in open(f)]
    print(os.path.basename(f), "->", len(rows), "rows | first: pred=%s gold=%s" %
          (rows[0]["pred_label"], rows[0]["gold_label"]))

$ /usr/bin/python3 src/run_generation.py --provider openai_compatible --model gemini-3.6-flash --base-url https://generativelanguage.googleapis.com/v1beta/openai/ --api-key-env GEMINI_API_KEY --limit 8

=== baseline1_plain_llm  (8 rows) -> results/runs/baseline1_plain_llm.jsonl ===
  [1/8] anx3-001: pred=high-risk gold=high-risk ok
  [2/8] anx3-002: pred=high-risk gold=high-risk ok
  [3/8] anx3-003: pred=high-risk gold=high-risk ok
  [4/8] anx3-004: pred=high-risk gold=high-risk ok
  [5/8] anx3-005: pred=high-risk gold=high-risk ok
  [6/8] anx3-006: pred=high-risk gold=high-risk ok
  [7/8] anx3-007: pred=high-risk gold=high-risk ok
  [8/8] anx3-008: pred=not-high-risk gold=not-high-risk PARSE?

=== baseline2_standard_rag  (8 rows) -> results/runs/baseline2_standard_rag.jsonl ===
  [1/8] anx3-001: pred=high-risk gold=high-risk ok
  [2/8] anx3-002: pred=high-risk gold=high-risk ok
  [3/8] anx3-003: pred=high-risk gold=high-risk ok
  [4/8] anx3-004: pred=high-risk gold=high-risk PARSE?
  

## 7. Judge + scoring
`stub` = offline and instant (dev). `vllm` = the real 70B judge on this GPU: the next cell starts a vLLM server (first run downloads the 70B, can take a while), scores against it, then stops it.

In [32]:
vllm_proc = subprocess.Popen([sys.executable,"-m","vllm.entrypoints.openai.api_server",
    "--model", JUDGE_MODEL, "--port", "8000",
    "--tensor-parallel-size", "2",        # split across both A6000s
    "--dtype", "auto",
    "--gpu-memory-utilization", "0.90",
    "--max-model-len", "8192"])
if JUDGE == "vllm":
    if not HAS_GPU:
        print("JUDGE=vllm but no GPU detected. Switch JUDGE to 'stub' or run on the GPU node.")
    else:
        subprocess.run([sys.executable,"-m","pip","install","-q","vllm"], check=False)
        import time, urllib.request
        vllm_proc = subprocess.Popen([sys.executable,"-m","vllm.entrypoints.openai.api_server",
            "--model", JUDGE_MODEL, "--port", "8000", "--dtype", "auto"])
        print("starting vLLM (first load downloads the 70B weights)...")
        for _ in range(180):
            try:
                urllib.request.urlopen("http://localhost:8000/v1/models", timeout=3)
                print("vLLM server is up"); break
            except Exception:
                time.sleep(10)
else:
    print("JUDGE=stub — no model needed; scoring runs offline and instant.")


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python3 -m pip install --upgrade pip


starting vLLM (first load downloads the 70B weights)...
(APIServer pid=6677) INFO 09-07 22:40:25 [api_utils.py:333] 
(APIServer pid=6677) INFO 09-07 22:40:25 [api_utils.py:333]        █     █     █▄   ▄█
(APIServer pid=6677) INFO 09-07 22:40:25 [api_utils.py:333]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.28.0
(APIServer pid=6677) INFO 09-07 22:40:25 [api_utils.py:333]   █▄█▀ █     █     █     █  model   Qwen/Qwen2.5-72B-Instruct-AWQ
(APIServer pid=6677) INFO 09-07 22:40:25 [api_utils.py:333]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=6677) INFO 09-07 22:40:25 [api_utils.py:333] 
(APIServer pid=6677) INFO 09-07 22:40:25 [api_utils.py:272] non-default args: {'model': 'Qwen/Qwen2.5-72B-Instruct-AWQ', 'max_model_len': 8192, 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.9}


(APIServer pid=6677) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


(APIServer pid=6677) INFO 09-07 22:40:26 [model.py:672] Resolved architecture: Qwen2ForCausalLM
(APIServer pid=6677) INFO 09-07 22:40:26 [model.py:1965] Using max model len 8192
(APIServer pid=6746) INFO 09-07 22:40:26 [api_utils.py:333] 
(APIServer pid=6746) INFO 09-07 22:40:26 [api_utils.py:333]        █     █     █▄   ▄█
(APIServer pid=6746) INFO 09-07 22:40:26 [api_utils.py:333]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.28.0
(APIServer pid=6746) INFO 09-07 22:40:26 [api_utils.py:333]   █▄█▀ █     █     █     █  model   Qwen/Qwen2.5-72B-Instruct-AWQ
(APIServer pid=6746) INFO 09-07 22:40:26 [api_utils.py:333]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=6746) INFO 09-07 22:40:26 [api_utils.py:333] 
(APIServer pid=6746) INFO 09-07 22:40:26 [api_utils.py:272] non-default args: {'model': 'Qwen/Qwen2.5-72B-Instruct-AWQ'}
(APIServer pid=6746) INFO 09-07 22:40:27 [model.py:672] Resolved architecture: Qwen2ForCausalLM
(APIServer pid=6746) INFO 09-07 22:40:27 [model.py:1965] Using max model len 327

Parse safetensors files: 100%|██████████| 11/11 [00:00<00:00, 18.41it/s]


...

[rank0]:[W907 22:46:37.110586764 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 243269632 bytes (free: 93978624, total: 50897289216).
[rank0]:[W907 22:46:37.153882285 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 973078528 bytes (free: 85590016, total: 50897289216).
[rank0]:[W907 22:46:37.221184329 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 268435456 bytes (free: 85590016, total: 50897289216).
[rank0]:[W907 22:46:37.235689701 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 1946157056 bytes (free: 152698880, total: 50897289216).
[rank0]:[W907 22:46:37.313885248 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 973078528 bytes (free: 135921664, total: 50897289216).
[rank0]:[W907 22:46:37.375825541 CUDACachingAllocato

(EngineCore pid=7634) INFO 09-07 22:46:51 [model_runner.py:380] Model loading took 38.77 GiB memory and 371.651462 seconds
(EngineCore pid=7634) INFO 09-07 22:46:51 [topk_topp_sampler.py:62] Using FlashInfer for top-p & top-k sampling.
(EngineCore pid=7634) INFO 09-07 22:47:02 [backends.py:1094] Using cache directory: /home/jovyan/.cache/vllm/torch_compile_cache/1d9e042724/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=7634) INFO 09-07 22:47:02 [backends.py:1155] Dynamo bytecode transform time: 10.75 s
(EngineCore pid=7634) INFO 09-07 22:47:08 [backends.py:393] Compiling a graph for compile range (1, 2048) takes 4.00 s
(EngineCore pid=7634) INFO 09-07 22:47:14 [backends.py:920] collected artifacts: 81 entries, 3 artifacts, 5049632 bytes total
(EngineCore pid=7634) INFO 09-07 22:47:14 [decorators.py:708] saved AOT compiled function to /home/jovyan/.cache/vllm/torch_compile_cache/torch_aot_compile/22b0ac182a157d87092b95aa884acdab06e3cca7e0b9bed3e170d42af26a7b90/rank_0_0/model

(EngineCore pid=7634) Process EngineCore:
(EngineCore pid=7634) Traceback (most recent call last):
(EngineCore pid=7634)   File "/usr/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
(EngineCore pid=7634)     self.run()
(EngineCore pid=7634)   File "/usr/lib/python3.11/multiprocessing/process.py", line 108, in run
(EngineCore pid=7634)     self._target(*self._args, **self._kwargs)
(EngineCore pid=7634)   File "/home/jovyan/.local/lib/python3.11/site-packages/vllm/v1/engine/core.py", line 1350, in run_engine_core
(EngineCore pid=7634)     raise e
(EngineCore pid=7634)   File "/home/jovyan/.local/lib/python3.11/site-packages/vllm/v1/engine/core.py", line 1315, in run_engine_core
(EngineCore pid=7634)     engine_core = EngineCoreProc(*args, engine_index=dp_rank, **kwargs)
(EngineCore pid=7634)                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
(EngineCore pid=7634)   File "/home/jovyan/.local/lib/python3.11/site-packages/vllm/tracing/otel.py", line 

(APIServer pid=6746) INFO 09-07 22:47:22 [utils.py:615] [shutdown] Process manager: send sigterm to process EngineCore


(APIServer pid=6746) Traceback (most recent call last):
(APIServer pid=6746)   File "<frozen runpy>", line 198, in _run_module_as_main
(APIServer pid=6746)   File "<frozen runpy>", line 88, in _run_code
(APIServer pid=6746)   File "/home/jovyan/.local/lib/python3.11/site-packages/vllm/entrypoints/openai/api_server.py", line 779, in <module>
(APIServer pid=6746)     uvloop.run(run_server(args))
(APIServer pid=6746)   File "/usr/local/lib/python3.11/dist-packages/uvloop/__init__.py", line 92, in run
(APIServer pid=6746)     return runner.run(wrapper())
(APIServer pid=6746)            ^^^^^^^^^^^^^^^^^^^^^
(APIServer pid=6746)   File "/usr/lib/python3.11/asyncio/runners.py", line 118, in run
(APIServer pid=6746)     return self._loop.run_until_complete(task)
(APIServer pid=6746)            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
(APIServer pid=6746)   File "uvloop/loop.pyx", line 1518, in uvloop.loop.Loop.run_until_complete
(APIServer pid=6746)   File "/usr/local/lib/python3.11/dist-packages/

In [33]:
score_args = [sys.executable, "src/run_scoring.py", "--judge", JUDGE]
if JUDGE == "vllm":
    score_args += ["--judge-model", JUDGE_MODEL, "--judge-base-url", JUDGE_BASE_URL]
sh(score_args)

if vllm_proc is not None:
    vllm_proc.terminate(); print("vLLM server stopped")

$ /usr/bin/python3 src/run_scoring.py --judge vllm --judge-model Qwen/Qwen2.5-72B-Instruct-AWQ --judge-base-url http://localhost:8000/v1
scoring runs in results/runs  ->  results/scoring
judge: vllm (Qwen/Qwen2.5-72B-Instruct-AWQ) @ http://localhost:8000/v1

[1/3] correctness
    agent_structured         acc=1.000  HR_f1=1.000  parse_fail=1
    baseline1_plain_llm      acc=1.000  HR_f1=1.000  parse_fail=1
    baseline2_standard_rag   acc=1.000  HR_f1=1.000  parse_fail=2

[2/3] faithfulness
  faithfulness: agent_structured (1 rows) judge=openai-compatible:Qwen/Qwen2.5-72B-Instruct-AWQ
  faithfulness: baseline2_standard_rag (8 rows) judge=openai-compatible:Qwen/Qwen2.5-72B-Instruct-AWQ

STDERR:
 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/jovyan/.local/lib/python3.11/site-packages/openai/_client.py", line 570, in _send_with_auth_retry
    response = super()._send_request(request, stream=stream, **kwargs)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/

## 8. Results — the four-bucket matrix
Correctness, faithfulness, then the right/wrong x faithful/unfaithful buckets. The off-diagonal rows are in results/scoring/buckets/*_offdiagonal.jsonl.

In [34]:
for f in ["results/scoring/correctness_summary.md",
          "results/scoring/faithfulness_summary.md",
          "results/scoring/buckets_summary.md"]:
    print("="*72); print(f); print("="*72)
    print(open(f).read() if os.path.exists(f) else "(not produced yet)")
    print()

results/scoring/correctness_summary.md
# Correctness (predicted label vs frozen Commission ground truth)

Positive class = high-risk. Accuracy plus per-class precision/recall/F1 because the set is imbalanced (frozen). Parse failures counted as incorrect and also shown separately.

| Config | n | Accuracy | HR precision | HR recall | HR F1 | Macro F1 | Parse fails |
|---|---|---|---|---|---|---|---|
| agent_structured | 1 | 1.000 | 1.000 | 1.000 | 1.000 | 0.500 | 1 |
| baseline1_plain_llm | 8 | 1.000 | 1.000 | 1.000 | 1.000 | 1.000 | 1 |
| baseline2_standard_rag | 8 | 1.000 | 1.000 | 1.000 | 1.000 | 1.000 | 2 |

## agent_structured

Confusion (high-risk positive): tp=1 fp=0 fn=0 tn=0

By edge-case type: none n=1 acc=1.000

By area: biometrics 1.000

## baseline1_plain_llm

Confusion (high-risk positive): tp=7 fp=0 fn=0 tn=1

By edge-case type: none n=8 acc=1.000

By area: biometrics 1.000

## baseline2_standard_rag

Confusion (high-risk positive): tp=7 fp=0 fn=0 tn=1

By edge-case type:

In [35]:
nvidia-smi

NameError: name 'nvidia' is not defined

In [ ]:
curl -s http://localhost:8000/v1/models